In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Modeling + preprocessing (we'll use these later)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings("ignore")

# 1. Load the dataset
df = pd.read_csv("medical_insurance.csv")

# 2. Take a quick look at the data
print("Shape of data:", df.shape)
display(df.head())

# 3. Check the target distribution for is_high_risk
print("\n'is_high_risk' value counts:")
print(df["is_high_risk"].value_counts())

print("\n'is_high_risk' proportion:")
print(df["is_high_risk"].value_counts(normalize=True))

Shape of data: (100000, 54)


,person_id,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,...,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,is_high_risk,had_major_procedure
0,75722,52,Female,North,Suburban,22700.0,Doctorate,Married,Retired,3,...,0,1,0,1,0,2,0,1,0,0
1,80185,79,Female,North,Urban,12800.0,No HS,Married,Employed,3,...,0,1,1,0,0,1,0,1,1,0
2,19865,68,Male,North,Rural,40700.0,HS,Married,Retired,5,...,0,0,1,1,0,2,1,0,1,0
3,76700,15,Male,North,Suburban,15600.0,Some College,Married,Self-employed,5,...,0,0,0,1,0,0,1,0,0,0
4,92992,53,Male,Central,Suburban,89600.0,Doctorate,Married,Self-employed,2,...,0,1,0,2,0,1,1,0,1,0



'is_high_risk' value counts:
is_high_risk
0    63219
1    36781
Name: count, dtype: int64

'is_high_risk' proportion:
is_high_risk
0    0.63219
1    0.36781
Name: proportion, dtype: float64


In [7]:
df.alcohol_freq = df.alcohol_freq.fillna('Never')

In [11]:
''' Build Feature Lists Automatically '''
# 1. Identify all columns in the dataset
all_cols = df.columns.tolist()

# 2. Columns we DO NOT want as inputs for the risk model
cols_to_drop = [
    "person_id",        # ID column
    "is_high_risk",     # TARGET variable - must be excluded
    "risk_score",       # not created yet but ensure excluded
    "risk_band",
    "annual_medical_cost",
    "annual_premium",
    "monthly_premium"
    "claims_count",
    "avg_claim_amount",
    "total_claims_paid"
    
]

# 3. Keep all other columns as potential features
feature_cols = [c for c in all_cols if c not in cols_to_drop]

# 4. Auto-detect NUMERIC columns (int or float)
numeric_cols = df[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()

# 5. Detect BINARY columns (0/1 only inside numeric group)
binary_cols = [
    c for c in numeric_cols
    if df[c].nunique() == 2 and set(df[c].unique()) <= {0, 1}
]

# 6. Remove binary columns from numeric list to leave TRUE numeric values
numeric_cols = [c for c in numeric_cols if c not in binary_cols]

# 7. Detect CATEGORICAL columns (dtype = object)
categorical_cols = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()

# 8. Show results
print("Total features for modeling:", len(feature_cols))
print("\nNumeric columns:", numeric_cols)
print("\nBinary columns:", binary_cols)
print("\nCategorical columns:", categorical_cols)

Total features for modeling: 47

Numeric columns: ['age', 'income', 'household_size', 'dependents', 'bmi', 'visits_last_year', 'hospitalizations_last_3yrs', 'days_hospitalized_last_3yrs', 'medication_count', 'systolic_bp', 'diastolic_bp', 'ldl', 'hba1c', 'deductible', 'copay', 'policy_term_years', 'policy_changes_last_2yrs', 'provider_quality', 'monthly_premium', 'claims_count', 'chronic_count', 'proc_imaging_count', 'proc_surgery_count', 'proc_physio_count', 'proc_consult_count', 'proc_lab_count']

Binary columns: ['hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease', 'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health', 'had_major_procedure']

Categorical columns: ['sex', 'region', 'urban_rural', 'education', 'marital_status', 'employment_status', 'smoker', 'alcohol_freq', 'plan_type', 'network_tier']


In [13]:
''' Train/Test Split + Preprocessing Pipeline '''
# 1. Define X and y
X = df[feature_cols]              # all predictor variables
y = df["is_high_risk"]            # target for risk prediction

# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # important for classification
)

# 3. Build preprocessing pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),                     # scale continuous vars
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),  # encode string vars
        ("bin", "passthrough", binary_cols)                          # leave binary vars unchanged
    ]
)

# Quick check
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Preprocessing pipeline ready.")

Train shape: (80000, 47)
Test shape: (20000, 47)
Preprocessing pipeline ready.


In [15]:
''' Logistic Regression (Baseline) '''
# Build full pipeline: preprocessing + model
baseline_logistic = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=500))
])

# Train the model
baseline_logistic.fit(X_train, y_train)

# Predictions
pred_proba = baseline_logistic.predict_proba(X_test)[:, 1]   # probability of high risk
pred = baseline_logistic.predict(X_test)                      # class 0/1 prediction

# Evaluation
print("Accuracy:", accuracy_score(y_test, pred))
print("F1 Score:", f1_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, pred_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, pred))
print("\nClassification Report:\n", classification_report(y_test, pred))

Accuracy: 0.9723
F1 Score: 0.9623334239869459
ROC-AUC: 0.9976465140074648

Confusion Matrix:
 [[12369   275]
 [  279  7077]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98     12644
           1       0.96      0.96      0.96      7356

    accuracy                           0.97     20000
   macro avg       0.97      0.97      0.97     20000
weighted avg       0.97      0.97      0.97     20000



In [19]:
''' LASSO LOGISTIC REGRESSION (L1) '''
from sklearn.model_selection import GridSearchCV

lasso_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=1000
    ))
])

lasso_params = {"clf__C": [0.01, 0.1, 1, 10]}

grid_lasso = GridSearchCV(
    lasso_pipe,
    lasso_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_lasso.fit(X_train, y_train)

# Best Lasso model
best_lasso = grid_lasso.best_estimator_

# Predictions
lasso_pred = best_lasso.predict(X_test)
lasso_proba = best_lasso.predict_proba(X_test)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, lasso_pred))
print("F1 Score:", f1_score(y_test, lasso_pred))
print("ROC-AUC:", roc_auc_score(y_test, lasso_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, lasso_pred))
print("\nClassification Report:\n", classification_report(y_test, lasso_pred))

Accuracy: 0.9726
F1 Score: 0.9627413652434049
ROC-AUC: 0.9976670388446467

Confusion Matrix:
 [[12372   272]
 [  276  7080]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98     12644
           1       0.96      0.96      0.96      7356

    accuracy                           0.97     20000
   macro avg       0.97      0.97      0.97     20000
weighted avg       0.97      0.97      0.97     20000



In [21]:
''' RIDGE LOGISTIC REGRESSION (L2) '''
ridge_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(penalty="l2", max_iter=1000))
])

ridge_params = {"clf__C": [0.01, 0.1, 1, 10]}

grid_ridge = GridSearchCV(
    ridge_pipe,
    ridge_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_ridge.fit(X_train, y_train)

# Best Ridge model
best_ridge = grid_ridge.best_estimator_

# Predictions
ridge_pred = best_ridge.predict(X_test)
ridge_proba = best_ridge.predict_proba(X_test)[:, 1]

# Evaluation (same style as baseline)
print("Accuracy:", accuracy_score(y_test, ridge_pred))
print("F1 Score:", f1_score(y_test, ridge_pred))
print("ROC-AUC:", roc_auc_score(y_test, ridge_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, ridge_pred))
print("\nClassification Report:\n", classification_report(y_test, ridge_pred))

Accuracy: 0.97235
F1 Score: 0.9624090816395894
ROC-AUC: 0.9976511694577005

Confusion Matrix:
 [[12368   276]
 [  277  7079]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98     12644
           1       0.96      0.96      0.96      7356

    accuracy                           0.97     20000
   macro avg       0.97      0.97      0.97     20000
weighted avg       0.97      0.97      0.97     20000



In [23]:
''' ELASTIC NET LOGISTIC REGRESSION (L1 + L2) '''
enet_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        max_iter=2000
    ))
])

enet_params = {
    "clf__C": [0.01, 0.1, 1],
    "clf__l1_ratio": [0.1, 0.5, 0.9]
}

grid_enet = GridSearchCV(
    enet_pipe,
    enet_params,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_enet.fit(X_train, y_train)

# Best Elastic Net model
best_enet = grid_enet.best_estimator_

# Predictions
enet_pred = best_enet.predict(X_test)
enet_proba = best_enet.predict_proba(X_test)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, enet_pred))
print("F1 Score:", f1_score(y_test, enet_pred))
print("ROC-AUC:", roc_auc_score(y_test, enet_proba))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, enet_pred))
print("\nClassification Report:\n", classification_report(y_test, enet_pred))

Accuracy: 0.97275
F1 Score: 0.962927691993742
ROC-AUC: 0.9976627059429262

Confusion Matrix:
 [[12377   267]
 [  278  7078]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98     12644
           1       0.96      0.96      0.96      7356

    accuracy                           0.97     20000
   macro avg       0.97      0.97      0.97     20000
weighted avg       0.97      0.97      0.97     20000



In [27]:
''' RANDOM FOREST CLASSIFIER '''
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
rf_pipe.fit(X_train, y_train)

# Predictions
rf_pred = rf_pipe.predict(X_test)
rf_proba = rf_pipe.predict_proba(X_test)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("F1 Score:", f1_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_proba))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))

Accuracy: 0.99405
F1 Score: 0.9919283727870853
ROC-AUC: 0.999871227881128

Confusion Matrix:
 [[12569    75]
 [   44  7312]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      1.00     12644
           1       0.99      0.99      0.99      7356

    accuracy                           0.99     20000
   macro avg       0.99      0.99      0.99     20000
weighted avg       0.99      0.99      0.99     20000



In [31]:
''' GRADIENT BOOSTING CLASSIFIER '''
from sklearn.ensemble import GradientBoostingClassifier

gb_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", GradientBoostingClassifier(
        n_estimators=400,      # number of boosting stages
        learning_rate=0.05,    # shrinkage
        max_depth=3,           # depth of individual trees
        random_state=42
    ))
])

# Train the model
gb_pipe.fit(X_train, y_train)

# Predictions
gb_pred = gb_pipe.predict(X_test)
gb_proba = gb_pipe.predict_proba(X_test)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, gb_pred))
print("F1 Score:", f1_score(y_test, gb_pred))
print("ROC-AUC:", roc_auc_score(y_test, gb_proba))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, gb_pred))
print("\nClassification Report:\n", classification_report(y_test, gb_pred))

Accuracy: 0.9999
F1 Score: 0.9998640380693405
ROC-AUC: 1.0

Confusion Matrix:
 [[12644     0]
 [    2  7354]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     12644
           1       1.00      1.00      1.00      7356

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



In [33]:
''' MODEL COMPARISON TABLE '''
results = []

def add_result(name, pred, proba):
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba)
    })

# Add each model's results
add_result("Baseline Logistic", pred, pred_proba)
add_result("Ridge Logistic", ridge_pred, ridge_proba)
add_result("Lasso Logistic", lasso_pred, lasso_proba)
add_result("Elastic Net Logistic", enet_pred, enet_proba)
add_result("Random Forest", rf_pred, rf_proba)
add_result("Gradient Boosting", gb_pred, gb_proba)

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Highlight the best model (max) for each metric
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: lightgreen' if v else '' for v in is_max]

styled_results = results_df.style.apply(highlight_max, subset=["Accuracy", "F1 Score", "ROC-AUC"])

styled_results

,Model,Accuracy,F1 Score,ROC-AUC
0,Baseline Logistic,0.972300,0.962333,0.997647
1,Ridge Logistic,0.972350,0.962409,0.997651
2,Lasso Logistic,0.972600,0.962741,0.997667
3,Elastic Net Logistic,0.972750,0.962928,0.997663
4,Random Forest,0.994050,0.991928,0.999871
5,Gradient Boosting,0.999900,0.999864,1.000000


In [37]:
# GENERATE predicted_risk_score USING BEST MODEL

best_model = gb_pipe   # <-- Replace if a different model was the best

# 1. Predict probability of high risk
df["predicted_risk_score"] = best_model.predict_proba(df[feature_cols])[:, 1]

# 2. Create predicted risk band
df["predicted_risk_band"] = pd.cut(
    df["predicted_risk_score"],
    bins=[0, 0.33, 0.66, 1.0],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

df[["risk_score", "predicted_risk_score", "predicted_risk_band"]].head()

,risk_score,predicted_risk_score,predicted_risk_band
0,0.5714,0.009868,Low
1,1.0000,0.999998,High
2,1.0000,1.000000,High
3,0.2857,0.000269,Low
4,0.8681,0.999994,High


In [43]:
# GET FINAL FEATURE NAMES AFTER PREPROCESSING

# 1. Get the preprocessing transformer
preprocessor = gb_pipe.named_steps["preprocess"]

# Extract columns from each transformer
num_features = numeric_cols
bin_features = binary_cols

# Extract one-hot encoded categorical column names
cat_encoder = preprocessor.named_transformers_["cat"]
cat_features = cat_encoder.get_feature_names_out(categorical_cols).tolist()

# Combine all final feature names in the correct pipeline order
final_feature_names = (
    num_features +          # scaled numerics
    cat_features +          # one-hot categorical
    bin_features            # passthrough binaries
)

len(final_feature_names), final_feature_names[:10]

(77,
 ['age',
  'income',
  'household_size',
  'dependents',
  'bmi',
  'visits_last_year',
  'hospitalizations_last_3yrs',
  'days_hospitalized_last_3yrs',
  'medication_count',
  'systolic_bp'])

In [45]:
''' EXTRACT FEATURE IMPORTANCES FROM GB MODEL '''

# Get the trained GB model
gb_model = gb_pipe.named_steps["clf"]

# Extract importances
importances = gb_model.feature_importances_

# Combine into a DataFrame
importance_df = pd.DataFrame({
    "feature": final_feature_names,
    "importance": importances
})

# Sort descending
importance_df = importance_df.sort_values(by="importance", ascending=False)

importance_df.head(20)

,feature,importance
0,age,4.466698e-01
20,chronic_count,3.881487e-01
51,smoker_Current,1.346663e-01
4,bmi,3.049510e-02
22,proc_surgery_count,2.012007e-05
18,monthly_premium,4.883535e-08
11,ldl,1.384579e-14
57,alcohol_freq_Weekly,0.000000e+00
56,alcohol_freq_Occasional,0.000000e+00
55,alcohol_freq_Never,0.000000e+00
